# Source Yellow Taxi Trip Data

In [0]:
# Create Notebook Parameter Widgets
dbutils.widgets.removeAll()
dbutils.widgets.text("ENVIRONMENT", "", "Environment")
dbutils.widgets.text("YEAR", "2020", "Trip Year")
dbutils.widgets.text("MONTH", "01", "Trip Month")
dbutils.widgets.dropdown("RESET_SINK", "False", ["True", "False"], "Reset Sink Location?")

In [0]:
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT")
YEAR = dbutils.widgets.get("YEAR")
MONTH = dbutils.widgets.get("MONTH")
RESET_SINK = (dbutils.widgets.get("RESET_SINK") == "True")


In [0]:
import requests
import uuid

from pyspark.sql.functions import *

In [0]:
SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DEST_PATH  = "abfss://{CONTAINER}@wbcdadpseudodlsa.dfs.core.windows.net/{ENVIRONMENT}"

CATALOG_NAME = 'raw_edl_dev_poc'
SCHEMA_NAME = ENVIRONMENT

INBOUND_PATH = DEST_PATH.format(CONTAINER = 'inbound', ENVIRONMENT = ENVIRONMENT)
VOLUME_URL = f'/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/inbound'
ODS_PATH = DEST_PATH.format(CONTAINER = 'ods', ENVIRONMENT = ENVIRONMENT)
TARGET_FILE_NAME = f"yellow_tripdata_{YEAR}-{MONTH}.parquet"
SOURCE = f"{SOURCE_URL}/{TARGET_FILE_NAME}"
TARGET = f"{VOLUME_URL}/{TARGET_FILE_NAME}"

TARGET_LOCATION = ODS_PATH + "/yellow_taxi_trips"
TARGET_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.yellow_taxi_trips" 

In [0]:
if RESET_SINK:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

    dbutils.fs.rm(TARGET_LOCATION, True)

In [0]:
# Create Inbound Location
dbutils.fs.mkdirs(INBOUND_PATH)

# Create Volume
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.inbound
  LOCATION '{INBOUND_PATH}'
  COMMENT 'Inbound Folder for {ENVIRONMENT}';
""")

In [0]:
# Download
print(f"Downloading from: {SOURCE}")

response = requests.get(SOURCE, stream=True, timeout=120)
response.raise_for_status()

with open(TARGET, "wb") as f:
    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
        f.write(chunk)

import os
size_mb = os.path.getsize(TARGET) / 1024 / 1024
print(f"Saved to {TARGET} ({size_mb:.1f} MB)")

In [0]:
df = spark.read.parquet(TARGET)

df = df.withColumns({
    "VendorID": col("VendorID").cast("int"),
    "passenger_count": col("passenger_count").cast("int"),
    "RatecodeID": col("RatecodeID").cast("int"),
    "payment_type": col("payment_type").cast("int"),
    "PULocationID": col("PULocationID").cast("int"),
    "DOLocationID": col("DOLocationID").cast("int"),
    "airport_fee": col("airport_fee").cast("double"),
    "trip_month": lit(MONTH).cast("int"),
    "trip_year": lit(YEAR).cast("int"),
    "source_execution_id": lit(str(uuid.uuid4())),
    "source_execution_dt": current_timestamp(),
    "source_file": lit(TARGET_FILE_NAME),
    "source_file_size": lit(size_mb).cast("int")
})

# or "append", "error", "ignore"
(df.write 
  .format("delta") 
  .partitionBy("trip_year", "trip_month")
  .option("path", TARGET_LOCATION) 
  .option("mergeSchema", "true") 
  .mode("overwrite")   
  .option("replaceWhere", f"trip_year = {int(YEAR)} AND trip_month = {int(MONTH)}")
  .saveAsTable(TARGET_TABLE))